# Mage-Flow trainer: data, latents and the remote GPU for the laptop GUI

Run **Settings** and **Install** first every session, then only the sections the machine needs:

| Section | Where | When |
| --- | --- | --- |
| 1. Upload the dataset | the Illustration Scrapping Studio server | once per training-layout export |
| 2. Models | every machine that caches or trains | once per machine (re-running only checks) |
| 3. Data | caching machine: **images**; training machine: **latent cache** | each session on a fresh machine |
| 4. Write the configs | every machine | after 2 and 3 |
| 5. Cache latents and captions | a cheap GPU box (e.g. the Studio server's A4000s) | once per dataset |
| 6. Upload the latent cache | the caching machine | after 5 |
| 7. Train | the training machine | |

The plan this follows: cache once where GPU time is cheap, upload only latents + captions + the caption-embedding
cache, and let the expensive training machine download just that (no images, no VAE, no text-encoder pass).
Tokens are never written into this notebook: log in with the cell in Install, or set `HF_TOKEN`.
See `docs/remote-training.md`.

In [ ]:
# ---- Settings: edit, then run this cell first every session ----
import os
WORK = "/workspace"                         # persistent storage here (Studio JupyterHub: "/home/jovyan")
TRAINER_DIR = f"{WORK}/mage-flow-trainer"
MODELS_DIR = f"{WORK}/models"
DATA_DIR = f"{WORK}/data/magetrail-v04-pilot"   # dataset root here: folders.csv + <group>/<artist>/ folders
OUTPUT_DIR = f"{WORK}/output"

DATASET_REPO = "RicemanT/magetrail-v04-pilot"           # private HF dataset: images + captions + training layout
CACHE_REPO = "RicemanT/magetrail-v04-pilot-latents"     # private HF dataset: latents + captions + caption cache, no images

BASE_MODEL_REPO = "mage-flow-community/Mage-Flow"      # the community copy (Microsoft took the original down)
TRANSFORMER_REPO = "RicemanT/MageTrail"
TRANSFORMER_FILE = "V0.3/MageTrail-V0.3.safetensors"    # the checkpoint the finetune starts from (architecture embedded)

GPUS = "0,1,2,3"                            # GPUs for caching here
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")   # faster up/downloads
MODEL_PATH = f"{MODELS_DIR}/mage-flow"
TRANSFORMER_PATH = f"{MODELS_DIR}/{TRANSFORMER_FILE}"

In [ ]:
# ---- Install: the trainer (once per machine), Hugging Face tools, login ----
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!git clone https://github.com/RicemanT/mage-flow-trainer || (cd mage-flow-trainer && git pull)
%cd {TRAINER_DIR}
!bash install.sh
!venv/bin/python -m pip install -q -r requirements-tracking.txt   # only if you use wandb / TensorBoard / Trackio
%pip install -q "huggingface_hub[hf_transfer]>=0.25"

from huggingface_hub import login, whoami
try:
    print("Hugging Face:", whoami()["name"])
except Exception:
    login()   # a token with write access; huggingface_hub stores it, this notebook never does

## 1. Upload the dataset (on the Studio server)

In the Studio: **Planner → 5. Download → Export training layout**. That folder (`folders.csv`, `subsets.toml`,
`mageflow-512.toml`, `mageflow-1024.toml`, `README.txt`) becomes the root of the dataset repo, and every
`<group>/<artist>` folder it lists goes beside it: images, `.txt` tags and `_nl.txt` captions. Folders the layout
does not list are not uploaded. The upload resumes if interrupted (it keeps its progress in
`<library>/images/.cache/huggingface/`).

In [ ]:
LIBRARY_IMAGES = "/home/jovyan/Artist-Collection-Dataset/images"
TRAINING_EXPORT = "/home/jovyan/Artist-Collection-Dataset/planner/exports/run-1/training"   # the export's folder

import csv
from huggingface_hub import HfApi
api = HfApi()
with open(f"{TRAINING_EXPORT}/folders.csv", encoding="utf-8") as fh:
    folders = [row["path"] for row in csv.DictReader(fh) if int(float(row.get("images") or 0))]
print(f"{len(folders)} artist folders listed")
api.create_repo(DATASET_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(repo_id=DATASET_REPO, repo_type="dataset", folder_path=TRAINING_EXPORT,
                  commit_message="Training layout (folders.csv, subsets.toml, configs)")
api.upload_large_folder(repo_id=DATASET_REPO, repo_type="dataset", folder_path=LIBRARY_IMAGES,
                        allow_patterns=[f"{path}/*" for path in folders],
                        ignore_patterns=["*_trainer.safetensors", "*.flagged", "*.tmp", ".*", "*/.*"])
print("Uploaded to https://huggingface.co/datasets/" + DATASET_REPO)

## 2. Models

The text encoder (Qwen3-VL-4B, with its tokenizer files) and Mage-VAE come from the Mage-Flow repository (its
8 GB base transformer is skipped); the transformer is your checkpoint (its architecture is embedded in the file, so no config is needed). Already-downloaded files are
only checked, so re-running is quick.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download(BASE_MODEL_REPO, local_dir=MODEL_PATH,
                  ignore_patterns=["transformer/*.safetensors", "assets/*"])   # ~9 GB: text encoder + VAE
hf_hub_download(TRANSFORMER_REPO, TRANSFORMER_FILE, local_dir=MODELS_DIR)
print("model_path       =", MODEL_PATH)
print("transformer_path =", TRANSFORMER_PATH)

## 3. Data: run ONE of the two cells

- **Images** (to cache latents): everything in `DATASET_REPO`.
- **Latent cache** (to train): `CACHE_REPO` only, no images. Training reads the latents (`dataset.source = "auto"`
  falls back to them when a folder has no images) and the captions beside them.

In [ ]:
# Images + captions + layout, for caching
snapshot_download(DATASET_REPO, repo_type="dataset", local_dir=DATA_DIR)
print("dataset in", DATA_DIR)

In [ ]:
# Latents + captions + caption cache + layout, for training (no images)
snapshot_download(CACHE_REPO, repo_type="dataset", local_dir=DATA_DIR)
print("latent cache in", DATA_DIR)

## 4. Write the configs for this machine

Fills the `@...@` placeholders of the exported `mageflow-*.toml` with this machine's paths and saves them as
`configs/magetrail/*.toml` in the trainer. For the 1024 stage, set `STAGE_512_CHECKPOINT` to the 512 stage's final
checkpoint first (a `.safetensors` file under `OUTPUT_DIR/<run_name>/`). Open the files in the GUI to adjust
epochs, learning rates or prompts; with jobs mode the GUI edits the config it sends.

In [ ]:
import glob, re
STAGE_512_CHECKPOINT = ""        # e.g. f"{OUTPUT_DIR}/magetrail-run1-512/<final checkpoint>.safetensors"

values = {"@MODEL_PATH@": MODEL_PATH, "@TRANSFORMER_PATH@": TRANSFORMER_PATH, "@DATA_DIR@": DATA_DIR,
          "@OUTPUT_DIR@": OUTPUT_DIR}
if STAGE_512_CHECKPOINT:
    values["@STAGE_512_CHECKPOINT@"] = STAGE_512_CHECKPOINT
target = f"{TRAINER_DIR}/configs/magetrail"
os.makedirs(target, exist_ok=True)
CONFIGS = {}
for source in sorted(glob.glob(f"{DATA_DIR}/mageflow-*.toml")):
    text = open(source, encoding="utf-8").read()
    for key, value in values.items():
        text = text.replace(key, value)
    name = os.path.basename(source)
    CONFIGS[name.removeprefix("mageflow-").removesuffix(".toml")] = f"{target}/{name}"
    with open(f"{target}/{name}", "w", encoding="utf-8") as fh:
        fh.write(text)
    left = sorted(set(re.findall(r"@[A-Z0-9_]+@", text)))
    print(f"{target}/{name}" + (f"   still to fill: {', '.join(left)}" if left else ""))

## 5. Cache latents and captions (needs images and models)

Encodes every listed image with Mage-VAE at each stage's resolution (`*_WWWWxHHHH_trainer.safetensors` beside the
image), then builds the caption-embedding cache (`caption_variations.sqlite` in `DATA_DIR`) with Qwen3-VL. Both are
reused by the training machine as they are; already-cached files are skipped, so an interrupted run resumes.
Leave out GPUs that other work (the Studio's image analysis) is using.

In [ ]:
for stage, config in CONFIGS.items():
    print(f"=== latents for {stage}")
    !cd {TRAINER_DIR} && venv/bin/python -m trainer.tools.cache_latents cache-config {config} --devices {GPUS}

In [ ]:
# Caption embeddings: one SQLite file shared by both stages. One process per GPU.
first = next(iter(CONFIGS.values()))
n = len([g for g in GPUS.split(",") if g.strip()])
!cd {TRAINER_DIR} && CUDA_VISIBLE_DEVICES={GPUS} venv/bin/accelerate launch --num_processes {n} -m trainer.training.train {first} --text-cache-only

## 6. Upload the latent cache

Uploads the latents, the `.txt`/`_nl.txt` captions, the caption cache and the layout files from `DATA_DIR`, never
the images, to `CACHE_REPO`. The first cell checks that every image has its latents for both stages and its
captions, so the training machine misses nothing.

In [ ]:
# Every listed image needs one latent per stage and its captions before the cache goes up.
import csv, glob, os
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
stages = len(CONFIGS)
missing_latents, missing_captions, images = [], [], 0
with open(f"{DATA_DIR}/folders.csv", encoding="utf-8") as fh:
    folders = [row["path"] for row in csv.DictReader(fh) if int(float(row.get("images") or 0))]
for folder in folders:
    for entry in os.scandir(f"{DATA_DIR}/{folder}"):
        stem, ext = os.path.splitext(entry.name)
        if ext.lower() not in IMAGE_EXT:
            continue
        images += 1
        if len(glob.glob(f"{DATA_DIR}/{glob.escape(folder)}/{glob.escape(stem)}_*_trainer.safetensors")) < stages:
            missing_latents.append(f"{folder}/{entry.name}")
        if not os.path.exists(f"{DATA_DIR}/{folder}/{stem}.txt"):
            missing_captions.append(f"{folder}/{entry.name}")
print(f"{images} images in {len(folders)} folders · {len(missing_latents)} without a latent for all {stages} stages"
      f" · {len(missing_captions)} without a .txt")
print("caption cache:", "present" if os.path.exists(f"{DATA_DIR}/caption_variations.sqlite") else "MISSING (run the caption cell)")
for label, items in (("no latents", missing_latents), ("no captions", missing_captions)):
    if items:
        print(f"  {label}: {items[:5]}{' ...' if len(items) > 5 else ''}")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(CACHE_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_large_folder(repo_id=CACHE_REPO, repo_type="dataset", folder_path=DATA_DIR,
                        allow_patterns=["*_trainer.safetensors", "*.txt", "*.csv", "*.toml", "*.json",
                                        "caption_variations.sqlite"],
                        ignore_patterns=[".*", "*/.*", "*.lock"])
print("Uploaded to https://huggingface.co/datasets/" + CACHE_REPO)

## 7. Train

**A. Jobs mode** (the GUI's Start Training runs here): leave the cell running and paste the printed link into the
GUI's Remote bar, then Connect. Load `configs/magetrail/mageflow-512.toml` in the GUI (or set the same paths:
`model_path`, `transformer_path`, `subsets_file`, `caption_cache_path`). Logs and graphs appear in the GUI; Stop /
Save now / Save & stop work from there.

In [ ]:
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote serve
# Own address instead of a Cloudflare tunnel, e.g. a RunPod proxy port:
# !cd {TRAINER_DIR} && venv/bin/python -m trainer.remote serve --tunnel none --url https://<pod-id>-8765.proxy.runpod.net

**B. Notebook mode:** the GUI only sends the config (Start Training), this cell receives it, the next one trains it.

In [ ]:
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote receive

In [ ]:
# The config the previous cell printed, or one of CONFIGS from section 4. Caches what is missing, then trains.
CONFIG = CONFIGS.get("512", "configs/remote/mageflow.toml")
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote run {CONFIG}
# Checkpoint on demand from another cell:  !touch {OUTPUT_DIR}/<run_name>/save   (or save_quit to stop)